## Old PCA

In [ ]:
# === 95% confidence ellipse ==============================================
def confidence_ellipse95(x: np.ndarray, y: np.ndarray, ax: plt.Axes, facecolor='none', **kwargs):
    if x.size < 2:
        return
    
    cov = np.cov(x, y)
    if np.linalg.det(cov) == 0:
        return
    
    chi2_val = chi2.ppf(0.95, df=2)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = eigvals.argsort()[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]
    
    width, height = 2 * np.sqrt(eigvals * chi2_val)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    
    theta = np.linspace(0, 2 * np.pi, 100)
    ellipse = np.array([width/2 * np.cos(theta), height/2 * np.sin(theta)])
    
    rot = np.array([[np.cos(np.radians(angle)), -np.sin(np.radians(angle))],
                    [np.sin(np.radians(angle)), np.cos(np.radians(angle))]])
    ellipse = rot @ ellipse
    ellipse[0] += x.mean()
    ellipse[1] += y.mean()
    
    ax.fill(ellipse[0], ellipse[1], facecolor=facecolor, **kwargs)

# === PERMDISP-like homogeneity of dispersion test ============================
def test_dispersion(df: pd.DataFrame, group_ser: pd.Series, title: str):
    """Test di omogeneità delle dispersioni (simile a PERMDISP)"""
    df = df.copy()
    group_ser = group_ser.copy()
    
    common_ids = df.index.intersection(group_ser.index)
    df = df.loc[common_ids]
    group_ser = group_ser.loc[common_ids]
    
    # Calcola i centroidi per ogni gruppo
    group_centroids = {}
    for group in group_ser.unique():
        members = df[group_ser == group]
        centroid = members.mean().values
        group_centroids[group] = centroid
    
    # Calcola le distanze dai centroidi
    distances_to_centroid = []
    group_labels = []
    
    for sample_id in df.index:
        group = group_ser[sample_id]
        centroid = group_centroids[group]
        dist = euclidean(df.loc[sample_id].values, centroid)
        distances_to_centroid.append(dist)
        group_labels.append(group)
    
    # Test ANOVA sulle distanze
    dist_by_group = pd.DataFrame({'Group': group_labels, 'Distance': distances_to_centroid})
    grouped = [g['Distance'].values for _, g in dist_by_group.groupby('Group')]
    
    f_stat, p_val = f_oneway(*grouped)
    
    print(f"\n[PERMDISP-like test] Homogeneity of dispersions – {title}")
    print(f"F-statistic: {f_stat:.4f}, p-value: {p_val:.4e}")

# === PERMANOVA analysis ======================================================
def run_permanova(df: pd.DataFrame, group_ser: pd.Series, title: str, save_prefix: str):
    """PERMANOVA analysis"""
    df = df.copy()
    group_ser = group_ser.copy()
    
    common_ids = df.index.intersection(group_ser.index)
    df = df.loc[common_ids]
    group_ser = group_ser.loc[common_ids]
    
    df_with_groups = df.copy()
    df_with_groups['Group'] = group_ser
    
    try:
        dist_matrix = DistanceMatrix.from_iterable(
            df.values, 
            metric=euclidean, 
            keys=df.index
        )
        
        result = permanova(dist_matrix, df_with_groups['Group'], permutations=999)
        
        print(f"\nPERMANOVA result for {title}:")
        print(result)
        
        # Salva i risultati
        os.makedirs("outputs/PCA", exist_ok=True)
        with open(f"outputs/PCA/{save_prefix}_permanova.txt", 'w') as f:
            f.write(f"PERMANOVA result for {title}:\n")
            f.write(str(result))
            
    except Exception as e:
        print(f"[PERMANOVA ERROR] {e}")

# === PCA plot ================================================================
def run_pca(df_scaled: pd.DataFrame, group_ser: pd.Series, title: str, save_prefix: str):
    """PCA analysis and plotting"""
    # Allinea gli indici
    common_ids = df_scaled.index.intersection(group_ser.index)
    df_scaled = df_scaled.loc[common_ids]
    group_ser = group_ser.loc[common_ids]
    
    # PCA
    pca = PCA()
    pcs = pca.fit_transform(df_scaled.values)
    
    # DataFrame con PC e gruppi
    pcdf = pd.DataFrame(pcs[:, :2], index=df_scaled.index, columns=["PC1", "PC2"])
    pcdf = pcdf.join(group_ser, how="left")

    # Loadings
    loadings = pca.components_.T   # coefficenti delle variabili
    # feature_names = feat_df.loc[df_scaled.columns, "Compound"]
    # feature_names = feature_names.where(feature_names.notna(), df_scaled.columns)
    pc1_loadings = pd.Series(loadings[:, 0], index=df_scaled.columns)
    pc2_loadings = pd.Series(loadings[:, 1], index=df_scaled.columns)
    
    print("\n[Loadings] Most affecting variables:")
    print("PC1:")
    print(pc1_loadings.abs().sort_values(ascending=False).head(10))
    print("\nPC2:")
    print(pc2_loadings.abs().sort_values(ascending=False).head(10))
    
    # Colori per i gruppi
    groups = sorted(pcdf["Group"].dropna().unique())
    palette = sns.color_palette("deep", n_colors=len(groups))
    col_map = {g: palette[i] for i, g in enumerate(groups)}
    
    # Plot PCA
    sns.set_theme(style="white", context="paper")
    fig, ax = plt.subplots(figsize=(6.5, 5.5), facecolor='white')
    fig.patch.set_edgecolor('black')
    fig.patch.set_linewidth(1)
    
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1)
    
    # Scatter plot
    sns.scatterplot(
        data=pcdf, 
        x="PC1", 
        y="PC2", 
        hue="Group", 
        palette=col_map,
        s=80, 
        edgecolor="black", 
        linewidth=0.5, 
        alpha=0.9,
        ax=ax,
        legend=True
    )
    
    # Ellissi di confidenza
    for group, sub in pcdf.groupby("Group", dropna=False):
        if pd.isna(group):
            continue
        color = col_map.get(group, "grey")
        confidence_ellipse95(
            sub.PC1.values, 
            sub.PC2.values, 
            ax,
            facecolor=color, 
            alpha=0.15, 
            edgecolor="none"
        )
    
    # Etichette e formattazione
    ev = pca.explained_variance_ratio_ * 100
    ax.set_title(title, fontsize=9, pad=6)
    ax.set_xlabel(f"PC1 ({ev[0]:.1f}%)", fontsize=8)
    ax.set_ylabel(f"PC2 ({ev[1]:.1f}%)", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(True, linestyle="--", linewidth=0.3, alpha=0.5)
    ax.axis("equal")
    
    ax.legend(
        title="Group",
        bbox_to_anchor=(1.05, 1),
        loc='best',
        borderaxespad=0,
        fontsize=7,
        title_fontsize=8,
        frameon=False
    )
    
    sns.despine()
    plt.tight_layout()
    
    # Salva i plot
    os.makedirs("outputs", exist_ok=True)
    plt.savefig(f"outputs/PCA/{save_prefix}_pca2D.jpg", dpi=600, bbox_inches='tight')
    plt.savefig(f"outputs/PCA/{save_prefix}_pca2D.svg", bbox_inches='tight')
    plt.show()

    # Biplot
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    # scatter dei campioni
    sns.scatterplot(
        data=pcdf, x="PC1", y="PC2", hue="Group",
        palette=col_map, s=70, edgecolor="black", linewidth=0.5, alpha=0.9, ax=ax, legend=False
    )
    
    # aggiungi frecce delle variabili
    scale_factor = np.sqrt(pca.explained_variance_[:2]) * 5  # moltiplica per 3 per visibilità

    for i, var in enumerate(df_scaled.columns):
        ax.arrow(0, 0,
             loadings[i,0]*scale_factor[0],
             loadings[i,1]*scale_factor[1],
             color='black', alpha=0.6, head_width=0.05, length_includes_head=True)
        ax.text(loadings[i,0]*scale_factor[0]*1.1,
            loadings[i,1]*scale_factor[1]*1.1,
            var, color='black', fontsize=7)
    
    ax.set_xlabel(f"PC1 ({ev[0]:.1f}%)", fontsize=8)
    ax.set_ylabel(f"PC2 ({ev[1]:.1f}%)", fontsize=8)
    ax.set_title(f"{title} – Biplot", fontsize=9)
    ax.grid(True, linestyle="--", linewidth=0.3, alpha=0.5)
    ax.axhline(0, color="grey", lw=0.5)
    ax.axvline(0, color="grey", lw=0.5)
    
    plt.tight_layout()
    plt.savefig(f"outputs/PCA/{save_prefix}_biplot.jpg", dpi=600, bbox_inches='tight')
    plt.savefig(f"outputs/PCA/{save_prefix}_biplot.svg", bbox_inches='tight')
    plt.show()
    
    
    # Plot varianza spiegata
    fig, ax = plt.subplots(figsize=(5.5, 3.2))
    ax.bar(range(1, len(ev) + 1), ev, color="grey")
    ax.plot(range(1, len(ev) + 1), np.cumsum(ev), color="black", lw=1, label="Cumulative")
    ax.set_xlabel("Principal Component", fontsize=8)
    ax.set_ylabel("Explained Variance (%)", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.legend(loc="upper right", fontsize=7)
    ax.set_title("Explained Variance", fontsize=9)
    
    sns.despine()
    plt.tight_layout()
    
    plt.savefig(f"outputs/PCA/{save_prefix}_explained_variance.jpg", dpi=600, bbox_inches='tight')
    plt.savefig(f"outputs/PCA/{save_prefix}_explained_variance.svg", bbox_inches='tight')
    plt.show()
    
    # Test di dispersione
    test_dispersion(
        pd.DataFrame(pcs[:, :2], index=df_scaled.index, columns=["PC1", "PC2"]),
        group_ser,
        title
    )
    
    # PERMANOVA
    run_permanova(
        pd.DataFrame(pcs[:, :2], index=df_scaled.index, columns=["PC1", "PC2"]),
        group_ser,
        title,
        save_prefix
    )

    def export_top_variables_csv(df_scaled, pc_loadings, comp_idx, filename, n_top=10):
        import re
        top_vars = pc_loadings.abs().sort_values(ascending=False).head(n_top).index
        df_export = df_scaled[top_vars].copy()
        df_export.insert(0, "Sample", df_export.index)

        df_export['SampleGroup'] = df_export['Sample'].apply(lambda x: re.sub(r'_\d+$', '', x))
        df_avg = df_export.groupby('SampleGroup').mean(numeric_only=True)
        df_avg.reset_index(inplace=True)
        avg_out_path = f"outputs/PCA/{filename.replace('.csv', '_avg.csv')}"
        df_avg.to_csv(avg_out_path, index=False)
        print(f"Saved: {avg_out_path}")

    # Export PC1
    export_top_variables_csv(df_scaled, pc1_loadings, 0, f"{save_prefix}_top_pc1.csv")
    # Export PC2
    export_top_variables_csv(df_scaled, pc2_loadings, 1, f"{save_prefix}_top_pc2.csv")

# === MAIN ====================================================================
if __name__ == '__main__':
    print("=== PCA Hot Infuses ===")
    run_pca(hot_scaled, grp_map, 'PCA – Hot infuses', 'hot')

    print("\n=== PCA Cold Infuses ===")
    run_pca(cold_scaled, grp_map, 'PCA – Cold infuses', 'cold')

## Old Heatmap


In [ ]:
def run_heatmap(
    df_scaled: pd.DataFrame, 
    group_ser: pd.Series, 
    title: str, 
    save_prefix: str,
    figsize=(5, 5), 
    show=True
):
    """Crea una heatmap clusterizzata"""
    df_scaled = df_scaled.copy()
    group_ser = group_ser.copy()
    
    # Allinea gli indici
    common = df_scaled.index.intersection(group_ser.index)
    df_scaled = df_scaled.loc[common]
    group_ser = group_ser.loc[common]
    
    # Aggiungi i gruppi e ordina
    df_with_groups = df_scaled.copy()
    df_with_groups['Group'] = group_ser
    df_with_groups.sort_values('Group', inplace=True)
    
    group_labels = df_with_groups.pop('Group')
    data = df_with_groups.T
    
    # Filtra i feature con almeno un valore non zero
    data = data[(data != 0).any(axis=1)]
    
    # Prova ad applicare i nomi dei composti se disponibili
    try:
        if 'Compound' in feat_df.columns:
            compound_names = feat_df.set_index(feat_df.index)['Compound'].reindex(data.index)
            # Sostituisci NaN con l'indice originale
            compound_names = compound_names.fillna(pd.Series(data.index, index=data.index))
            data.index = compound_names.values
    except Exception as e:
        print(f"[WARN] Compound names not applied → {e}")
    
    # Colori per i gruppi
    groups = sorted(group_labels.unique())
    palette = sns.color_palette("deep", len(groups))
    lut = {g: palette[i] for i, g in enumerate(groups)}
    col_colors = group_labels.map(lut)
    
    # Adatta le dimensioni in base al numero di feature e campioni
    n_features, n_samples = data.shape
    
    # Calcola dimensioni dinamiche
    min_width = max(figsize[0], n_samples * 0.3 + 4)
    min_height = max(figsize[1], n_features * 0.1 + 4)
    adjusted_figsize = (min(min_width, 20), min(min_height, 16))
    
    # Adatta la dimensione dei font in base al numero di elementi
    label_fontsize = max(4, min(8, 200 / max(n_features, n_samples)))
    
    # Crea la heatmap
    sns.set_theme(style="white", context="paper")
    
    g = sns.clustermap(
        data,
        cmap="vlag",
        figsize=adjusted_figsize,
        row_cluster=True,
        col_cluster=True,
        col_colors=col_colors,
        dendrogram_ratio=(.1, .1),
        cbar_pos=(.92, .3, .02, .4),
        xticklabels=True,
        yticklabels=True,
        cbar_kws={"label": "Z-score"},
        linewidths=0
    )
    
    g.ax_heatmap.tick_params(labelsize=label_fontsize)
    
    # Gestione etichette x con rotazione adattiva
    plt.setp(g.ax_heatmap.get_xticklabels(), 
             rotation=90 if n_samples > 10 else 45, 
             ha='right', 
             fontsize=label_fontsize)
    
    plt.setp(g.ax_heatmap.get_yticklabels(), 
             fontsize=label_fontsize)
    
    # Legenda per i gruppi
    legend_handles = [Patch(facecolor=lut[g], label=g) for g in groups]
    g.ax_col_dendrogram.legend(
        handles=legend_handles,
        title="Group",
        loc="upper left",
        bbox_to_anchor=(1.02, 1),
        fontsize=max(6, label_fontsize),
        title_fontsize=max(7, label_fontsize + 1),
        frameon=False
    )
    
    # Usa subplots_adjust invece di tight_layout per un controllo migliore
    g.fig.subplots_adjust(right=0.85, bottom=0.15, top=0.9)
    
    # Salva i plot
    os.makedirs("outputs/HEAT-MAP", exist_ok=True)
    g.fig.savefig(f"outputs/HEAT-MAP/{save_prefix}_heatmap.jpg", dpi=600, bbox_inches='tight')
    g.fig.savefig(f"outputs/HEAT-MAP/{save_prefix}_heatmap.svg", bbox_inches='tight')

    if show:
        plt.show()
    
    plt.close()


if __name__ == '__main__':
    print("\n=== Heatmap Hot Infuses ===")
    run_heatmap(hot_scaled, grp_map, 'Heatmap – Hot infuses', 'hot')
    
    print("\n=== Heatmap Cold Infuses ===")
    run_heatmap(cold_scaled, grp_map, 'Heatmap – Cold infuses', 'cold')

In [ ]:
### PLS-DA 28-10-2025
# ============================================================================
# PLS-DA ANALYSIS WITH OPTIMAL COMPONENT SELECTION
# ============================================================================
"""
Partial Least Squares Discriminant Analysis (PLS-DA) with:
- AUTOMATIC OPTIMAL COMPONENT SELECTION (avoiding overfitting)
- Adaptive cross-validation strategy
- Robust permutation testing (1000 permutations)
- Comprehensive visualization and reporting
"""

output_dir = Path(".") / "outputs" / "PLS-DA"
output_dir.mkdir(parents=True, exist_ok=True)

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 9)
plt.rcParams['font.size'] = 10

# ============================================================================
# STEP 1: LOAD PREPROCESSED DATA
# ============================================================================

print("\n" + "="*80)
print("LOADING GLOBALLY-SCALED PREPROCESSED DATA")
print("="*80)

hot_scaled = pd.read_csv("outputs/hot_scaled.csv", index_col=0)
cold_scaled = pd.read_csv("outputs/cold_scaled.csv", index_col=0)

METADATA = Path(r"C:\Users\david\OneDrive - Università degli Studi di Parma\PhD\Projects\Coffee_leaf_infuses\data\gcms\volatile_infuses_metadata.tsv")
meta_raw = pd.read_csv(METADATA, sep='\t')

print(f"\n✓ HOT dataset: {hot_scaled.shape[0]} samples × {hot_scaled.shape[1]} metabolites")
print(f"✓ COLD dataset: {cold_scaled.shape[0]} samples × {cold_scaled.shape[1]} metabolites")

grp_map = meta_raw.set_index('ATTRIBUTE_Sample')['ATTRIBUTE_Group']

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def select_cv_strategy(y: np.ndarray, dataset_name: str = "Dataset") -> tuple:
    """Automatically select CV strategy based on smallest class size."""
    unique, counts = np.unique(y, return_counts=True)
    min_class = counts.min()
    
    print(f"\n{dataset_name}:")
    print(f"  Class distribution: {dict(zip(unique, counts))}")
    print(f"  Min class size: {min_class}")
    
    if min_class < 3:
        cv = LeaveOneOut()
        cv_name = "LOO"
        cv_info = "Leave-One-Out Cross-Validation"
    elif min_class < 5:
        cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        cv_name = "3-Fold"
        cv_info = "3-Fold Stratified K-Fold"
    elif min_class < 10:
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        cv_name = "5-Fold"
        cv_info = "5-Fold Stratified K-Fold"
    else:
        cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
        cv_name = "10x5-Fold"
        cv_info = "Repeated 10 × 5-Fold Stratified K-Fold"
    
    print(f"  → Strategy: {cv_info}")
    return cv, cv_name, cv_info


def calculate_vip_scores(X: np.ndarray, y: np.ndarray, 
                        pls_model: PLSRegression) -> np.ndarray:
    """Calculate Variable Importance in Projection (VIP) scores."""
    n_features = X.shape[1]
    n_components = pls_model.n_components
    
    W = pls_model.x_weights_
    T = pls_model.x_scores_
    SS = np.sum(T ** 2, axis=0)
    
    VIP = np.zeros(n_features)
    for j in range(n_features):
        vip_sum = np.sum((W[j, :] ** 2) * SS)
        VIP[j] = np.sqrt(n_features * vip_sum / np.sum(SS))
    
    return VIP


def calculate_q2(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Calculate Q² (cross-validated R²)."""
    residuals = y_true - y_pred
    press = np.sum(residuals ** 2)
    tss = np.sum((y_true - y_true.mean()) ** 2)
    
    if tss > 0:
        q2 = 1 - (press / tss)
    else:
        q2 = np.nan
    
    return q2


def find_optimal_components(X: np.ndarray, y: np.ndarray,
                           cv_splitter,
                           max_comp: int = 5,
                           dataset_name: str = "Dataset") -> dict:
    """
    Find optimal number of components based on CV Q² performance.
    
    Returns optimal n_components that balances:
    - High Q² (predictive power)
    - Low overfitting (R²_train - Q²_cv difference)
    - Model parsimony (fewer components is better)
    
    Parameters:
    -----------
    X : np.ndarray
        Feature matrix (already scaled)
    y : np.ndarray
        Class labels
    cv_splitter : cross-validator
        CV strategy
    max_comp : int
        Maximum components to test (default: 5)
    dataset_name : str
        Name for logging
        
    Returns:
    --------
    dict with component analysis results
    """
    
    print(f"\n{'─'*80}")
    print(f"OPTIMAL COMPONENT SELECTION")
    print(f"{'─'*80}")
    print(f"Testing 1 to {max_comp} components...")
    
    n_samples = X.shape[0]
    # Limit max_comp: use at most n_samples/3 to avoid overfitting
    max_comp = min(max_comp, max(1, n_samples // 3))
    
    component_results = []
    
    for n_comp in range(1, max_comp + 1):
        cv_q2_list = []
        cv_r2_list = []
        
        # CV loop
        for train_idx, test_idx in cv_splitter.split(X, y):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            
            y_train_2d = y_train.reshape(-1, 1)
            pls_cv = PLSRegression(n_components=n_comp, max_iter=1000)
            pls_cv.fit(X_train, y_train_2d)
            
            y_pred_cv = pls_cv.predict(X_test).ravel()
            q2 = calculate_q2(y_test.astype(float), y_pred_cv)
            r2 = pls_cv.score(X_test, y_test.reshape(-1, 1))
            
            cv_q2_list.append(q2)
            cv_r2_list.append(r2)
        
        cv_q2_mean = np.nanmean(cv_q2_list)
        cv_r2_mean = np.nanmean(cv_r2_list)
        
        # Train on full data to check for overfitting
        y_2d = y.reshape(-1, 1)
        pls_full = PLSRegression(n_components=n_comp, max_iter=1000)
        pls_full.fit(X, y_2d)
        
        y_pred_full = pls_full.predict(X).ravel()
        r2_train = pls_full.score(X, y_2d)
        q2_train = calculate_q2(y.astype(float), y_pred_full)
        
        # Overfitting indicator: difference between training and CV performance
        overfit_indicator = r2_train - cv_q2_mean
        
        component_results.append({
            'n_components': n_comp,
            'cv_q2_mean': cv_q2_mean,
            'cv_r2_mean': cv_r2_mean,
            'r2_train': r2_train,
            'q2_train': q2_train,
            'overfit_indicator': overfit_indicator
        })
        
        # Print results with interpretation
        if overfit_indicator < 0.1:
            status = "✅ Good"
        elif overfit_indicator < 0.2:
            status = "⚠️  Moderate overfitting"
        else:
            status = "❌ High overfitting"
        
        print(f"\n  {n_comp} comp: CV_Q²={cv_q2_mean:.4f}, Train_R²={r2_train:.4f}, "
              f"Δ={overfit_indicator:.4f} {status}")
    
    component_df = pd.DataFrame(component_results)
    
    # Select optimal: highest Q² that is not overfitting too much
    # Penalize high overfitting
    scores = component_df['cv_q2_mean'].values - 0.5 * component_df['overfit_indicator'].values
    optimal_idx = np.argmax(scores)
    optimal_n_comp = component_df.iloc[optimal_idx]['n_components']
    
    print(f"\n{'─'*80}")
    print(f"✅ OPTIMAL COMPONENTS SELECTED: {int(optimal_n_comp)}")
    print(f"{'─'*80}")
    print(f"  Reason: Best balance between Q² performance and avoiding overfitting")
    print(f"  Q²: {component_df.iloc[optimal_idx]['cv_q2_mean']:.4f}")
    print(f"  Overfitting indicator: {component_df.iloc[optimal_idx]['overfit_indicator']:.4f}")
    
    return {
        'optimal_n_components': int(optimal_n_comp),
        'component_df': component_df,
        'all_results': component_results
    }


def plsda_single_extraction(df_scaled: pd.DataFrame, group_series: pd.Series,
                           title: str, save_prefix: str,
                           n_perm: int = 1000,
                           force_n_components: int = None) -> dict:
    """
    Comprehensive PLS-DA analysis with optimal component selection.
    
    Parameters:
    -----------
    df_scaled : pd.DataFrame
        Globally-scaled feature matrix
    group_series : pd.Series
        Group assignment for each sample
    title : str
        Title for plots and reports
    save_prefix : str
        Prefix for saved files
    n_perm : int
        Number of permutations (default: 1000)
    force_n_components : int
        If provided, use this specific number instead of finding optimal
        
    Returns:
    --------
    dict with comprehensive results
    """
    
    print(f"\n{'█'*80}")
    print(f"█  {title.upper()}")
    print(f"█  (MetaboAnalyst-Compliant Workflow: Global Scaling)")
    print(f"{'█'*80}")
    
    # Data preparation
    common_ids = df_scaled.index.intersection(group_series.index)
    X = df_scaled.loc[common_ids].values
    
    group_labels = group_series.loc[common_ids].values
    le = LabelEncoder()
    y = le.fit_transform(group_labels)
    
    n_samples, n_features = X.shape
    
    print(f"\n📊 Dataset Information:")
    print(f"  Samples: {n_samples}")
    print(f"  Metabolites: {n_features}")
    print(f"  ⚠️  Data is ALREADY globally scaled (preprocessing step)")
    
    unique, counts = np.unique(y, return_counts=True)
    print(f"\n  Class distribution:")
    for cls_name, cls_label, n in zip(le.classes_, unique, counts):
        print(f"    {cls_name}: {n} samples (class {cls_label})")
    
    # Select CV strategy
    cv_splitter, cv_name, cv_info = select_cv_strategy(y, dataset_name=title)
    
    # Find optimal components
    if force_n_components is not None:
        n_components = force_n_components
        print(f"\n⚠️  Using FORCED n_components={n_components}")
    else:
        component_analysis = find_optimal_components(
            X, y, cv_splitter, max_comp=5, dataset_name=title
        )
        n_components = component_analysis['optimal_n_components']
    
    # ====================================================================
    # CROSS-VALIDATION (with optimal components)
    # ====================================================================
    
    print(f"\n{'─'*80}")
    print(f"CROSS-VALIDATION ({cv_name}) with {n_components} components")
    print(f"{'─'*80}")
    
    cv_metrics = {
        'accuracy': [],
        'r2': [],
        'q2': []
    }
    
    fold_idx = 0
    for train_idx, test_idx in cv_splitter.split(X, y):
        fold_idx += 1
        
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        y_train_2d = y_train.reshape(-1, 1)
        pls_cv = PLSRegression(n_components=n_components, max_iter=1000)
        pls_cv.fit(X_train, y_train_2d)
        
        y_pred_cv = pls_cv.predict(X_test).ravel()
        y_pred_binary = np.round(y_pred_cv).astype(int)
        y_pred_binary = np.clip(y_pred_binary, 0, y.max())
        
        acc = accuracy_score(y_test, y_pred_binary)
        r2 = pls_cv.score(X_test, y_test.reshape(-1, 1))
        q2 = calculate_q2(y_test.astype(float), y_pred_cv)
        
        cv_metrics['accuracy'].append(acc)
        cv_metrics['r2'].append(r2)
        cv_metrics['q2'].append(q2)
    
    acc_mean = np.nanmean(cv_metrics['accuracy'])
    acc_std = np.nanstd(cv_metrics['accuracy'])
    r2_mean = np.nanmean(cv_metrics['r2'])
    r2_std = np.nanstd(cv_metrics['r2'])
    q2_mean = np.nanmean(cv_metrics['q2'])
    q2_std = np.nanstd(cv_metrics['q2'])
    
    print(f"\n✅ Completed {fold_idx} folds")
    print(f"\n📈 CV Results ({n_components} components):")
    print(f"  Accuracy:  {acc_mean:.4f} ± {acc_std:.4f}")
    print(f"  R²:        {r2_mean:.4f} ± {r2_std:.4f}")
    print(f"  Q² (pred): {q2_mean:.4f} ± {q2_std:.4f}")
    
    # ====================================================================
    # FINAL MODEL ON FULL DATA
    # ====================================================================
    
    print(f"\n{'─'*80}")
    print(f"FINAL MODEL (trained on all data)")
    print(f"{'─'*80}")
    
    y_2d = y.reshape(-1, 1)
    
    pls_final = PLSRegression(n_components=n_components, max_iter=1000)
    pls_final.fit(X, y_2d)
    
    y_pred_full = pls_final.predict(X).ravel()
    y_pred_full_binary = np.round(y_pred_full).astype(int)
    y_pred_full_binary = np.clip(y_pred_full_binary, 0, y.max())
    
    r2_final = pls_final.score(X, y_2d)
    q2_final = calculate_q2(y.astype(float), y_pred_full)
    acc_final = accuracy_score(y, y_pred_full_binary)
    
    print(f"\n✅ Final Model Performance:")
    print(f"  R² (training):  {r2_final:.4f}")
    print(f"  Q² (CV mean):   {q2_mean:.4f}")
    print(f"  Accuracy:       {acc_final:.4f}")
    
    # Check overfitting
    overfit_diff = r2_final - q2_mean
    if overfit_diff > 0.15:
        print(f"\n  ⚠️  WARNING: Possible overfitting detected!")
        print(f"      R²_train - Q²_cv = {overfit_diff:.4f} (threshold: 0.15)")
    else:
        print(f"\n  ✅ No significant overfitting (R²_train - Q²_cv = {overfit_diff:.4f})")
    
    # ====================================================================
    # PERMUTATION TEST
    # ====================================================================
    
    print(f"\n{'─'*80}")
    print(f"PERMUTATION TEST (N={n_perm})")
    print(f"{'─'*80}")
    
    perm_r2 = []
    perm_q2 = []
    
    np.random.seed(42)
    
    for perm_idx in range(n_perm):
        y_perm = np.random.permutation(y)
        y_perm_2d = y_perm.reshape(-1, 1)
        
        pls_perm = PLSRegression(n_components=n_components, max_iter=1000)
        pls_perm.fit(X, y_perm_2d)
        
        y_pred_perm = pls_perm.predict(X).ravel()
        r2_perm = pls_perm.score(X, y_perm_2d)
        q2_perm = calculate_q2(y_perm.astype(float), y_pred_perm)
        
        perm_r2.append(r2_perm)
        perm_q2.append(q2_perm)
        
        if (perm_idx + 1) % 250 == 0:
            print(f"  {perm_idx + 1}/{n_perm} permutations completed")
    
    perm_r2 = np.array(perm_r2)
    perm_q2 = np.array(perm_q2)
    
    p_value_r2 = (np.sum(perm_r2 >= r2_final) + 1) / (n_perm + 1)
    p_value_q2 = (np.sum(perm_q2 >= q2_final) + 1) / (n_perm + 1)
    
    print(f"\n✅ Permutation Test Results:")
    print(f"\n  R² (Goodness of Fit):")
    print(f"    Observed: {r2_final:.4f}")
    print(f"    Permuted ≥ observed: {np.sum(perm_r2 >= r2_final)}")
    print(f"    P-value: {p_value_r2:.6f}")
    
    print(f"\n  Q² (Predictive Power):")
    print(f"    Observed: {q2_final:.4f}")
    print(f"    Permuted ≥ observed: {np.sum(perm_q2 >= q2_final)}")
    print(f"    P-value: {p_value_q2:.6f}")
    
    if p_value_q2 < 0.001:
        sig_text = "✅ HIGHLY SIGNIFICANT (p < 0.001)"
    elif p_value_q2 < 0.01:
        sig_text = "✅ VERY SIGNIFICANT (p < 0.01)"
    elif p_value_q2 < 0.05:
        sig_text = "✅ SIGNIFICANT (p < 0.05)"
    else:
        sig_text = "❌ NOT SIGNIFICANT (p ≥ 0.05)"
    
    print(f"\n  {sig_text}")
    
    # ====================================================================
    # VIP SCORES
    # ====================================================================
    
    print(f"\n{'─'*80}")
    print(f"VARIABLE IMPORTANCE IN PROJECTION (VIP)")
    print(f"{'─'*80}")
    
    vip = calculate_vip_scores(X, y, pls_final)
    
    vip_df = pd.DataFrame({
        'Metabolite': df_scaled.columns,
        'VIP': vip
    }).sort_values('VIP', ascending=False).reset_index(drop=True)
    
    n_vip_1 = (vip_df['VIP'] >= 1.0).sum()
    
    print(f"\n✅ VIP Score Summary:")
    print(f"  Metabolites with VIP ≥ 1.0: {n_vip_1} ({100*n_vip_1/len(vip_df):.1f}%)")
    print(f"  Mean VIP: {vip_df['VIP'].mean():.4f}")
    print(f"  Median VIP: {vip_df['VIP'].median():.4f}")
    
    print(f"\n  Top 10 Discriminative Metabolites:")
    for idx, (_, row) in enumerate(vip_df.head(10).iterrows(), 1):
        print(f"    {idx:2d}. {row['Metabolite']:<15} VIP = {row['VIP']:.4f}")
    
    significant_mets = vip_df[vip_df['VIP'] >= 1.0]
    
    # ====================================================================
    # CONFUSION MATRIX & CLASSIFICATION REPORT
    # ====================================================================
    
    cm = confusion_matrix(y, y_pred_full_binary)
    
    print(f"\n{'─'*80}")
    print(f"CLASSIFICATION PERFORMANCE")
    print(f"{'─'*80}")
    
    print(f"\nConfusion Matrix:")
    print(cm)
    
    print(f"\nClassification Report:")
    print(classification_report(y, y_pred_full_binary, target_names=le.classes_))
    
    # ====================================================================
    # VISUALIZATION
    # ====================================================================
    
    print(f"\n{'─'*80}")
    print(f"GENERATING VISUALIZATIONS")
    print(f"{'─'*80}")
    
    fig = plt.figure(figsize=(18, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.3)
    
    # Plot 1: CV Metrics
    ax1 = fig.add_subplot(gs[0, 0])
    metrics_names = ['Accuracy', 'R²', 'Q²']
    means = [acc_mean, r2_mean, q2_mean]
    stds = [acc_std, r2_std, q2_std]
    colors = ['#3498DB', '#2ECC71', '#F39C12']
    
    bars = ax1.bar(metrics_names, means, yerr=stds, capsize=8, color=colors,
                    edgecolor='black', linewidth=1.5, alpha=0.8)
    ax1.set_ylabel('Score', fontweight='bold')
    ax1.set_ylim([0, 1.05])
    ax1.grid(True, alpha=0.3, axis='y')
    ax1.set_title(f'CV Metrics ({n_components} comp)\n{cv_name}', fontweight='bold', fontsize=10)
    
    for bar, mean, std in zip(bars, means, stds):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + std,
                f'{mean:.3f}±{std:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    
    # Plot 2: Component Selection (if available)
    ax2 = fig.add_subplot(gs[0, 1])
    if force_n_components is None and 'component_analysis' in locals():
        comp_df = component_analysis['component_df']
        ax2.plot(comp_df['n_components'], comp_df['cv_q2_mean'], 
                marker='o', linewidth=2.5, markersize=8, color='#E74C3C', label='CV Q²')
        ax2.axvline(n_components, color='#27AE60', linestyle='--', linewidth=2, 
                   label=f'Selected: {n_components}', alpha=0.7)
        ax2.fill_between(comp_df['n_components'], 
                        comp_df['cv_q2_mean'] - comp_df['overfit_indicator'],
                        comp_df['cv_q2_mean'],
                        alpha=0.2, color='red', label='Overfitting gap')
        ax2.set_xlabel('# Components', fontweight='bold')
        ax2.set_ylabel('Q² (CV)', fontweight='bold')
        ax2.grid(True, alpha=0.3)
        ax2.legend(fontsize=8)
        ax2.set_title('Component Selection (Elbow Method)', fontweight='bold', fontsize=10)
        ax2.set_xticks(comp_df['n_components'])
    else:
        ax2.text(0.5, 0.5, f'Using forced\nn_components={n_components}', 
                ha='center', va='center', fontsize=12, fontweight='bold',
                transform=ax2.transAxes)
        ax2.axis('off')
    
    # Plot 3: R² vs Q² Model Quality
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.scatter([r2_final], [q2_mean], s=300, color='#E74C3C', marker='*',
               edgecolor='black', linewidth=2, zorder=5, label='Observed')
    ax3.axhline(0.5, color='gray', linestyle='--', alpha=0.5, linewidth=1, label='Q²=0.5')
    ax3.axhline(0.9, color='green', linestyle='--', alpha=0.5, linewidth=1, label='Q²=0.9')
    ax3.set_xlabel('R² (Fit)', fontweight='bold')
    ax3.set_ylabel('Q² (Prediction)', fontweight='bold')
    ax3.set_xlim([0, 1.05])
    ax3.set_ylim([0, 1.05])
    ax3.grid(True, alpha=0.3)
    ax3.legend(fontsize=8)
    ax3.set_title('Model Quality', fontweight='bold', fontsize=10)
    
    # Plot 4: Permutation Test R²
    ax4 = fig.add_subplot(gs[1, 0])
    ax4.hist(perm_r2, bins=40, alpha=0.7, color='#95A5A6', edgecolor='black', linewidth=0.5)
    ax4.axvline(r2_final, color='#E74C3C', linestyle='--', linewidth=2.5,
               label=f'Observed R²={r2_final:.3f}')
    ax4.set_xlabel('Permutation R²', fontweight='bold')
    ax4.set_ylabel('Frequency', fontweight='bold')
    ax4.set_title(f'Perm. Test R² (p={p_value_r2:.4f})', fontweight='bold', fontsize=10)
    ax4.legend(fontsize=8)
    ax4.grid(True, alpha=0.3, axis='y')
    
    # Plot 5: Permutation Test Q²
    ax5 = fig.add_subplot(gs[1, 1])
    ax5.hist(perm_q2, bins=40, alpha=0.7, color='#E8DAEF', edgecolor='black', linewidth=0.5)
    ax5.axvline(q2_final, color='#9B59B6', linestyle='--', linewidth=2.5,
               label=f'Observed Q²={q2_final:.3f}')
    ax5.set_xlabel('Permutation Q²', fontweight='bold')
    ax5.set_ylabel('Frequency', fontweight='bold')
    ax5.set_title(f'Perm. Test Q² (p={p_value_q2:.4f})', fontweight='bold', fontsize=10)
    ax5.legend(fontsize=8)
    ax5.grid(True, alpha=0.3, axis='y')
    
    # Plot 6: Confusion Matrix
    ax6 = fig.add_subplot(gs[1, 2])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax6,
               xticklabels=le.classes_, yticklabels=le.classes_,
               cbar_kws={'label': 'Count'})
    ax6.set_xlabel('Predicted', fontweight='bold')
    ax6.set_ylabel('True', fontweight='bold')
    ax6.set_title('Confusion Matrix', fontweight='bold', fontsize=10)
    
    # Plot 7: Top VIP Scores
    ax7 = fig.add_subplot(gs[2, :2])
    top_vip_n = min(20, len(vip_df))
    top_vip = vip_df.head(top_vip_n)
    colors_vip = ['#E74C3C' if x >= 1.0 else '#BDC3C7' for x in top_vip['VIP']]
    
    ax7.barh(range(len(top_vip)), top_vip['VIP'], color=colors_vip,
            edgecolor='black', linewidth=0.7)
    ax7.set_yticks(range(len(top_vip)))
    ax7.set_yticklabels(top_vip['Metabolite'], fontsize=8)
    ax7.set_xlabel('VIP Score', fontweight='bold')
    ax7.set_title(f'Top {top_vip_n} Discriminative Metabolites', fontweight='bold', fontsize=10)
    ax7.axvline(1.0, color='#E74C3C', linestyle='--', linewidth=1.5, alpha=0.7, label='VIP=1.0')
    ax7.legend(fontsize=8)
    ax7.grid(True, alpha=0.3, axis='x')
    ax7.invert_yaxis()
    
    # Plot 8: VIP Distribution
    ax8 = fig.add_subplot(gs[2, 2])
    ax8.hist(vip_df['VIP'], bins=30, alpha=0.7, color='#16A085', edgecolor='black', linewidth=0.5)
    ax8.axvline(1.0, color='#E74C3C', linestyle='--', linewidth=2, label='VIP=1.0')
    ax8.axvline(vip_df['VIP'].mean(), color='#F39C12', linestyle='-', linewidth=2, 
               label=f'Mean={vip_df["VIP"].mean():.2f}')
    ax8.set_xlabel('VIP Score', fontweight='bold')
    ax8.set_ylabel('Frequency', fontweight='bold')
    ax8.set_title('VIP Distribution', fontweight='bold', fontsize=10)
    ax8.legend(fontsize=8)
    ax8.grid(True, alpha=0.3, axis='y')
    
    fig.suptitle(f'PLS-DA Analysis: {title}\n({n_components} components, {n_samples} samples, {len(le.classes_)} groups)\n[MetaboAnalyst Workflow: Global Scaling + Optimal Components]',
                fontsize=13, fontweight='bold', y=0.995)
    
    plt.savefig(output_dir / f"{save_prefix}_plsda_analysis.png", dpi=300, bbox_inches='tight')
    plt.savefig(output_dir / f"{save_prefix}_plsda_analysis.svg", format='svg', bbox_inches='tight')
    print(f"✓ Saved: {save_prefix}_plsda_analysis.png")
    plt.close()
    
    # ====================================================================
    # EXPORT RESULTS
    # ====================================================================
    
    print(f"\n{'─'*80}")
    print(f"EXPORTING RESULTS")
    print(f"{'─'*80}")
    
    vip_df.to_csv(output_dir / f"{save_prefix}_vip_scores.csv", index=False)
    print(f"✓ VIP scores: {save_prefix}_vip_scores.csv")
    
    significant_mets.to_csv(output_dir / f"{save_prefix}_significant_metabolites_vip_ge1.csv", index=False)
    print(f"✓ Significant metabolites ({len(significant_mets)}): {save_prefix}_significant_metabolites_vip_ge1.csv")
    
    cv_df = pd.DataFrame(cv_metrics)
    cv_df.to_csv(output_dir / f"{save_prefix}_cv_metrics_all_folds.csv", index=False)
    print(f"✓ CV metrics: {save_prefix}_cv_metrics_all_folds.csv")
    
    perm_df = pd.DataFrame({
        'Permutation_R2': perm_r2,
        'Permutation_Q2': perm_q2
    })
    perm_df.to_csv(output_dir / f"{save_prefix}_permutation_test_results.csv", index=False)
    print(f"✓ Permutation test: {save_prefix}_permutation_test_results.csv")
    
    summary_dict = {
        'Dataset': [title],
        'Workflow': ['MetaboAnalyst (Global Scaling + Optimal Components)'],
        'N_Samples': [n_samples],
        'N_Metabolites': [n_features],
        'N_Components': [n_components],
        'CV_Strategy': [cv_info],
        'N_Classes': [len(le.classes_)],
        'CV_Accuracy_Mean': [acc_mean],
        'CV_Accuracy_Std': [acc_std],
        'CV_R2_Mean': [r2_mean],
        'CV_R2_Std': [r2_std],
        'CV_Q2_Mean': [q2_mean],
        'CV_Q2_Std': [q2_std],
        'Final_R2': [r2_final],
        'Final_Q2': [q2_final],
        'Final_Accuracy': [acc_final],
        'Overfit_Indicator': [overfit_diff],
        'Perm_P_Value_R2': [p_value_r2],
        'Perm_P_Value_Q2': [p_value_q2],
        'N_Permutations': [n_perm],
        'N_Significant_Metabolites_VIP_GE1': [len(significant_mets)],
        'Percent_Significant': [100*len(significant_mets)/len(vip_df)],
        'Significance_Q2': [sig_text]
    }
    
    summary_df = pd.DataFrame(summary_dict)
    summary_df.to_csv(output_dir / f"{save_prefix}_summary_report.csv", index=False)
    print(f"✓ Summary: {save_prefix}_summary_report.csv")
    
    print(f"\n✅ All results exported to {output_dir}/")
    
    return {
        'title': title,
        'n_components': n_components,
        'n_samples': n_samples,
        'n_metabolites': n_features,
        'n_classes': len(le.classes_),
        'classes': le.classes_,
        'cv_info': cv_info,
        'cv_accuracy_mean': acc_mean,
        'cv_accuracy_std': acc_std,
        'cv_r2_mean': r2_mean,
        'cv_r2_std': r2_std,
        'cv_q2_mean': q2_mean,
        'cv_q2_std': q2_std,
        'final_r2': r2_final,
        'final_q2': q2_final,
        'final_accuracy': acc_final,
        'overfit_indicator': overfit_diff,
        'p_value_r2': p_value_r2,
        'p_value_q2': p_value_q2,
        'n_permutations': n_perm,
        'vip_df': vip_df,
        'significant_mets': significant_mets,
        'confusion_matrix': cm,
        'label_encoder': le,
        'perm_r2': perm_r2,
        'perm_q2': perm_q2
    }


# ============================================================================
# RUN ANALYSIS ON HOT AND COLD DATASETS
# ============================================================================

print("\n\n" + "█"*80)
print("█  HOT EXTRACTION ANALYSIS - WITH OPTIMAL COMPONENT SELECTION")
print("█"*80)

plsda_hot = plsda_single_extraction(
    hot_scaled, grp_map,
    'Hot Infuses', 'hot',
    n_perm=1000
)

print("\n\n" + "█"*80)
print("█  COLD EXTRACTION ANALYSIS - WITH OPTIMAL COMPONENT SELECTION")
print("█"*80)

plsda_cold = plsda_single_extraction(
    cold_scaled, grp_map,
    'Cold Infuses', 'cold',
    n_perm=1000
)

# ============================================================================
# COMPARATIVE SUMMARY: HOT vs COLD
# ============================================================================

print("\n\n" + "█"*80)
print("█  COMPARATIVE SUMMARY: HOT vs COLD")
print("█"*80)

comparison_data = {
    'Metric': [
        'Samples',
        'Metabolites',
        'Groups',
        'Components (Optimal)',
        'CV Strategy',
        '',
        'CV Accuracy (mean±std)',
        'CV R² (mean±std)',
        'CV Q² (mean±std)',
        '',
        'Final R² (training)',
        'Final Q² (CV)',
        'Final Accuracy',
        'Overfitting Indicator',
        '',
        'Perm. p-value (R²)',
        'Perm. p-value (Q²)',
        'Significance',
        '',
        'N Significant Metabolites (VIP≥1.0)',
        '% Significant',
    ],
    'HOT': [
        plsda_hot['n_samples'],
        plsda_hot['n_metabolites'],
        plsda_hot['n_classes'],
        plsda_hot['n_components'],
        plsda_hot['cv_info'],
        '',
        f"{plsda_hot['cv_accuracy_mean']:.4f}±{plsda_hot['cv_accuracy_std']:.4f}",
        f"{plsda_hot['cv_r2_mean']:.4f}±{plsda_hot['cv_r2_std']:.4f}",
        f"{plsda_hot['cv_q2_mean']:.4f}±{plsda_hot['cv_q2_std']:.4f}",
        '',
        f"{plsda_hot['final_r2']:.4f}",
        f"{plsda_hot['final_q2']:.4f}",
        f"{plsda_hot['final_accuracy']:.4f}",
        f"{plsda_hot['overfit_indicator']:.4f}",
        '',
        f"{plsda_hot['p_value_r2']:.6f}",
        f"{plsda_hot['p_value_q2']:.6f}",
        "✅ SIG" if plsda_hot['p_value_q2'] < 0.05 else "❌ NS",
        '',
        len(plsda_hot['significant_mets']),
        f"{100*len(plsda_hot['significant_mets'])/len(plsda_hot['vip_df']):.1f}%",
    ],
    'COLD': [
        plsda_cold['n_samples'],
        plsda_cold['n_metabolites'],
        plsda_cold['n_classes'],
        plsda_cold['n_components'],
        plsda_cold['cv_info'],
        '',
        f"{plsda_cold['cv_accuracy_mean']:.4f}±{plsda_cold['cv_accuracy_std']:.4f}",
        f"{plsda_cold['cv_r2_mean']:.4f}±{plsda_cold['cv_r2_std']:.4f}",
        f"{plsda_cold['cv_q2_mean']:.4f}±{plsda_cold['cv_q2_std']:.4f}",
        '',
        f"{plsda_cold['final_r2']:.4f}",
        f"{plsda_cold['final_q2']:.4f}",
        f"{plsda_cold['final_accuracy']:.4f}",
        f"{plsda_cold['overfit_indicator']:.4f}",
        '',
        f"{plsda_cold['p_value_r2']:.6f}",
        f"{plsda_cold['p_value_q2']:.6f}",
        "✅ SIG" if plsda_cold['p_value_q2'] < 0.05 else "❌ NS",
        '',
        len(plsda_cold['significant_mets']),
        f"{100*len(plsda_cold['significant_mets'])/len(plsda_cold['vip_df']):.1f}%",
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n" + comparison_df.to_string(index=False))

comparison_df.to_csv(output_dir / "comparison_hot_vs_cold_optimal_components.csv", index=False)
print(f"\n✓ Comparison saved: comparison_hot_vs_cold_optimal_components.csv")

# ============================================================================
# SIGNIFICANT METABOLITES ANALYSIS
# ============================================================================

print("\n" + "="*80)
print("SIGNIFICANT METABOLITES ANALYSIS")
print("="*80)

hot_sig = set(plsda_hot['significant_mets']['Metabolite'].astype(str))
cold_sig = set(plsda_cold['significant_mets']['Metabolite'].astype(str))
common_sig = hot_sig.intersection(cold_sig)
only_hot = hot_sig - cold_sig
only_cold = cold_sig - hot_sig

print(f"\n📊 Significant Metabolites (VIP ≥ 1.0):")
print(f"  HOT:         {len(hot_sig)}")
print(f"  COLD:        {len(cold_sig)}")
print(f"  COMMON:      {len(common_sig)}")
print(f"  ONLY HOT:    {len(only_hot)}")
print(f"  ONLY COLD:   {len(only_cold)}")

if len(common_sig) > 0:
    print(f"\n✅ Common Significant Metabolites ({len(common_sig)}):")
    for i, met in enumerate(sorted(list(common_sig))[:30], 1):
        hot_vip = plsda_hot['vip_df'][plsda_hot['vip_df']['Metabolite'] == met]['VIP'].values[0]
        cold_vip = plsda_cold['vip_df'][plsda_cold['vip_df']['Metabolite'] == met]['VIP'].values[0]
        print(f"  {i:2d}. {met:<15} | HOT VIP={hot_vip:.3f} | COLD VIP={cold_vip:.3f}")
    if len(common_sig) > 30:
        print(f"  ... and {len(common_sig) - 30} more")

if len(only_hot) > 0:
    print(f"\n🔴 Only in HOT ({len(only_hot)}):")
    for i, met in enumerate(sorted(list(only_hot))[:20], 1):
        vip = plsda_hot['vip_df'][plsda_hot['vip_df']['Metabolite'] == met]['VIP'].values[0]
        print(f"  {i:2d}. {met:<15} VIP={vip:.3f}")
    if len(only_hot) > 20:
        print(f"  ... and {len(only_hot) - 20} more")

if len(only_cold) > 0:
    print(f"\n🔵 Only in COLD ({len(only_cold)}):")
    for i, met in enumerate(sorted(list(only_cold))[:20], 1):
        vip = plsda_cold['vip_df'][plsda_cold['vip_df']['Metabolite'] == met]['VIP'].values[0]
        print(f"  {i:2d}. {met:<15} VIP={vip:.3f}")
    if len(only_cold) > 20:
        print(f"  ... and {len(only_cold) - 20} more")

common_detail = []
for met in common_sig:
    hot_vip = plsda_hot['vip_df'][plsda_hot['vip_df']['Metabolite'] == met]['VIP'].values[0]
    cold_vip = plsda_cold['vip_df'][plsda_cold['vip_df']['Metabolite'] == met]['VIP'].values[0]
    common_detail.append({
        'Metabolite': met,
        'HOT_VIP': hot_vip,
        'COLD_VIP': cold_vip,
        'VIP_Ratio_COLD_HOT': cold_vip / hot_vip if hot_vip > 0 else np.nan
    })

common_detail_df = pd.DataFrame(common_detail).sort_values('VIP_Ratio_COLD_HOT', ascending=False)
common_detail_df.to_csv(output_dir / "common_metabolites_detailed_optimal_components.csv", index=False)

venn_data = {
    'Category': ['COMMON', 'ONLY_HOT', 'ONLY_COLD'],
    'N_Metabolites': [len(common_sig), len(only_hot), len(only_cold)]
}
venn_df = pd.DataFrame(venn_data)
venn_df.to_csv(output_dir / "significant_metabolites_venn_data_optimal_components.csv", index=False)

print(f"\n✓ Metabolite comparison exported")

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "="*80)
print("✅ PLS-DA ANALYSIS COMPLETE (OPTIMAL COMPONENTS)")
print("="*80)
print(f"\n📁 All results saved in: {output_dir}/")
print(f"\n📊 Workflow Used: MetaboAnalyst (Global Scaling + Automatic Component Selection)")
print(f"   - Data pre-scaled before analysis ✓")
print(f"   - CV on scaled data (no per-fold re-scaling) ✓")
print(f"   - Optimal components selected based on Q² and overfitting detection ✓")
print(f"   - Permutation test on identically-scaled data ✓")
print(f"\n🔍 Component Selection Results:")
print(f"   - HOT dataset: {plsda_hot['n_components']} components")
print(f"   - COLD dataset: {plsda_cold['n_components']} components")
print(f"\n⚠️  Overfitting Indicators (R²_train - Q²_cv):")
print(f"   - HOT: {plsda_hot['overfit_indicator']:.4f}")
print(f"   - COLD: {plsda_cold['overfit_indicator']:.4f}")
print("\n" + "="*80 + "\n")